### IMPORT Modelos Regresión

# --- CORE ---
import numpy as np
import pandas as pd

# --- MODELOS REGRESIÓN ---
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# --- BOOSTING ---
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# --- VALIDACIÓN ---
from sklearn.model_selection import cross_val_score

# --- MÉTRICAS REGRESIÓN ---
# r2
# neg_root_mean_squared_error → RMSE
# neg_mean_absolute_error     → MAE
# neg_mean_squared_error      → MSE
# neg_mean_absolute_percentage_error → MAPE


### BaseLines para todos los modelos (hay que elegir 4 o 5 como mucho para probar)

In [ ]:
cv_reg = 5 # CV

modelos_regresion = {

    "Linear Regression": LinearRegression(
        n_jobs=-1
    ),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=None,        # baseline: sin limitar profundidad
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,      # mismo criterio que clasificación
        max_depth=3,           # mismo criterio
        random_state=42,
        n_jobs=-1
    ),

    "KNN": KNeighborsRegressor(
        n_neighbors=5,         # baseline clásico
        weights="uniform",
        n_jobs=-1
    ),

    "SVM": SVR(
        kernel="rbf",          # baseline estándar
        C=1.0,
        gamma="scale"
    ),

    "XGBoost": XGBRegressor(
        n_estimators=100,      # mismo criterio
        learning_rate=0.1,     # mismo criterio
        max_depth=3,           # mismo criterio: boosting → árboles poco profundos
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=100,      # mismo criterio
        learning_rate=0.1,     # mismo criterio
        num_leaves=31,         # mismo criterio
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),

    "CatBoost": CatBoostRegressor(
        iterations=100,        # equivalente a n_estimators
        learning_rate=0.1,
        depth=3,
        random_state=42,
        verbose=False          # silencia logs
    )
}

#── EVALUACIÓN REGRESIÓN
print("\n" + "=" * 55)
print("BASELINE REGRESIÓN")
print("=" * 55)
for nombre, modelo in modelos_regresion.items():
    scores_r2   = cross_val_score(modelo, X_scaled, y, cv=cv_reg, scoring="r2")
    scores_rmse = cross_val_score(modelo, X_scaled, y, cv=cv_reg,
                                  scoring="neg_root_mean_squared_error")
    print(f"{nombre:<22} R²: {scores_r2.mean():.4f} ± {scores_r2.std():.4f} "
          f"| RMSE: {(-scores_rmse.mean()):.4f} ± {scores_rmse.std():.4f}")


In [ ]:
# Elige aquí la métrica que quieres evaluar
# opciones: "r2", "neg_root_mean_squared_error", "neg_mean_absolute_error",
#           "neg_mean_squared_error", "neg_mean_absolute_percentage_error"
metrica_reg = "r2"

print("\n" + "=" * 55)
print(f"BASELINE REGRESIÓN — MÉTRICA: {metrica_reg}")
print("=" * 55)

for nombre, modelo in modelos_regresion.items():
    scores = cross_val_score(modelo, X_scaled, y, cv=cv_reg, scoring=metrica_reg)

    # Si la métrica es negativa (RMSE, MAE, MSE, MAPE), la invertimos
    if metrica_reg.startswith("neg_"):
        mean_score = -scores.mean()
        std_score  = scores.std()
    else:
        mean_score = scores.mean()
        std_score  = scores.std()

    print(f"{nombre:<22} {metrica_reg}: {mean_score:.4f} ± {std_score:.4f}")
